In [1]:
import random
from collections import defaultdict

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
# Dataset

text = """
artificial intelligence is transforming modern society.
it is used in healthcare finance education and transportation.
machine learning allows systems to improve automatically with experience.
data plays a critical role in training intelligent systems.
large datasets help models learn complex patterns.
deep learning uses multi layer neural networks.
neural networks are inspired by biological neurons.
each neuron processes input and produces an output.
training a neural network requires optimization techniques.
gradient descent minimizes the loss function.

natural language processing helps computers understand human language.
text generation is a key task in nlp.
language models predict the next word or character.
recurrent neural networks handle sequential data.
lstm and gru models address long term dependency problems.
however rnn based models are slow for long sequences.

transformer models changed the field of nlp.
they rely on self attention mechanisms.
attention allows the model to focus on relevant context.
transformers process data in parallel.
this makes training faster and more efficient.
modern language models are based on transformers.

education is being improved using artificial intelligence.
intelligent tutoring systems personalize learning.
automated grading saves time for teachers.
online education platforms use recommendation systems.
technology enhances the quality of learning experiences.

ethical considerations are important in artificial intelligence.
fairness transparency and accountability must be ensured.
ai systems should be designed responsibly.
data privacy and security are major concerns.
researchers continue to improve ai safety.

text generation models can create stories poems and articles.
they are used in chatbots virtual assistants and content creation.
generated text should be meaningful and coherent.
evaluation of text generation is challenging.
human judgement is often required.

continuous learning is essential in the field of ai.
research and innovation drive technological progress.
students should build strong foundations in mathematics.
programming skills are important for ai engineers.
practical experimentation enhances understanding.

"""


In [4]:
# Preprocessing

def preprocess_text(text):
    text = text.lower()
    text = text.replace('\n', ' ')
    tokens = text.split()
    return tokens

tokens = preprocess_text(text)


In [5]:
# Tokenization

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

token_list = tokenizer.texts_to_sequences([text])[0]


In [6]:
# Create Input Sequences

input_sequences = []

for i in range(1, len(token_list)):
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

max_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [7]:
#  Build LSTM Model

model = Sequential()
model.add(Embedding(total_words, 64, input_length=max_len-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
def sample_with_temperature(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

def generate_text(seed_text, next_words, model, max_len, temperature=0.8):
    for _ in range(next_words):
        sequence = tokenizer.texts_to_sequences([seed_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_len-1, padding='pre')

        preds = model.predict(sequence, verbose=0)[0]
        predicted_index = sample_with_temperature(preds, temperature)

        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                seed_text += " " + word
                break

    return seed_text

In [9]:
model.fit(X, y, epochs=150, verbose=1)

Epoch 1/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.0057 - loss: 5.2734
Epoch 2/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0330 - loss: 5.2587
Epoch 3/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0454 - loss: 5.1915
Epoch 4/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0395 - loss: 5.1057
Epoch 5/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0364 - loss: 5.0506
Epoch 6/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0324 - loss: 5.0470
Epoch 7/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0395 - loss: 5.0334
Epoch 8/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0517 - loss: 4.9473
Epoch 9/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0185 - loss: 5.0260
Epoch 10/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0490 - loss: 4.9452
Epoch 11/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0576 - loss: 4.9132
Epoch 12/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

In [10]:
# LSTM
seed_text = "artificial intelligence"
print(generate_text(seed_text, 20, model, max_len, temperature=0.6))
print(generate_text(seed_text, 40, model, max_len, temperature=0.8))
print(generate_text(seed_text, 60, model, max_len, temperature=1.0))

artificial intelligence is transforming modern society it is used in healthcare finance education and transportation machine learning allows systems to improve automatically
artificial intelligence is transforming modern society it is used in healthcare finance education and transportation machine learning allows systems to improve automatically with experience data plays a critical role in training intelligent systems large datasets datasets help models learn complex patterns patterns
artificial intelligence is transforming modern it society it is used in healthcare finance education and transportation machine learning allows systems to improve automatically with data plays a a networks in intelligent intelligent large datasets help models learn complex patterns deep learning uses multi layer neural networks neural networks are inspired by by each neuron neuron processes and produces an output a neural


In [13]:
# LSTM
seed_text = "artificial learning"
print(generate_text(seed_text, 20, model, max_len, temperature=0.6))
print(generate_text(seed_text, 40, model, max_len, temperature=0.8))
print(generate_text(seed_text, 60, model, max_len, temperature=1.0))

artificial learning intelligence is transforming modern society it is used in healthcare finance education and transportation machine learning allows systems to improve
artificial learning intelligence is transforming modern society it is used in healthcare finance education and transportation machine learning allows systems to improve automatically with experience data plays a critical role in training intelligent systems large datasets help models learn complex patterns deep
artificial learning intelligence is transforming modern society it is used in healthcare finance allows of systems learning to improve automatically with plays models a critical critical role in training intelligent tutoring systems large complex patterns deep deep learning uses multi multi layer neural networks neural networks are inspired by biological neurons each neuron processes input and produces an output training a neural


In [11]:
seed_text = "deep learning"
print(generate_text(seed_text, 20, model, max_len, temperature=0.6))
print(generate_text(seed_text, 40, model, max_len, temperature=0.8))
print(generate_text(seed_text, 60, model, max_len, temperature=1.0))

deep learning is transforming modern society it is used in healthcare finance education and transportation machine learning allows systems to improve automatically
deep learning is transforming modern society it is used in healthcare finance education is used in healthcare finance education and transportation machine learning allows systems to improve automatically with experience data plays a critical role in training intelligent systems large datasets learn
deep learning intelligence is is human society society education is healthcare finance education and transportation systems to automatically allows systems to improve automatically plays a critical role in training intelligent systems datasets large help models learn complex patterns deep learning neural multi layer neural networks neural networks are inspired by neurons each neuron neuron training input and produces an output training a
